In [1]:
# IMPORT LIBRARIES 

import xmltodict
import os
import hashlib

from rdflib import SKOS, RDF, Literal, Graph, Namespace, URIRef

In [2]:
# CONFIGs

# path to directory
directory_path = '../CBS-metadata/ODISSEI_Full_Export_20210924'

# output file name
ofile = "cbs-variables-thesaurus.ttl"

# define namespaces
var_ns = Namespace("https://portal.odissei-data.nl/data/cbs/variableThesaurus/")

In [3]:
graph = Graph()
graph.bind("cbsVar", var_ns)
graph.bind("skos", SKOS)
graph.bind("rdf", RDF)

In [4]:
def make_xml_dictionary(file):
    """
    Use the xmltodict library to create a dictionary from the xml file
    """
    xmlfile = open(file, 'r')
    xml_content = xmlfile.read()
    xml_dictionary = xmltodict.parse(xml_content)
    return xml_dictionary

In [5]:
def make_cbs_variable_thesaurus_id(var_ns):
    """
    Create ID of the thesaurus.
    """
    thesaurus_id = var_ns + "cbsVariableThesaurus"
    return thesaurus_id

In [6]:
def define_cbs_variable_thesaurus(thesaurus_id):
    """
    Create concept scheme of the CBS Variables Thesaurus.
    """
    graph.add((URIRef(thesaurus_id), RDF.type, SKOS.ConceptScheme))
    graph.add((URIRef(thesaurus_id), SKOS.prefLabel, Literal("CBS Variables Thesaurus")))
    return graph 

In [7]:
def make_variables_list(xml_dictionary):
    """
    Make a list out of the variables in the dictionary.
    """
    variables_list = xml_dictionary['Dataontwerpversies']['Versie']['Dataontwerp']['Contextvariabelen']['Contextvariabele']
    return variables_list

In [8]:
def make_broader_variable_id(var):
    """
    Create ID of the 'main' (or broader) variable.
    A 'v' has been added at the begging of the ID to indicate that
    we are refering to a variable.
    """
    broader_variable_id = var_ns + "v" + var['Variabele']['Id']
    return broader_variable_id

In [9]:
def make_narrower_variable_id(var):
    """
    Create ID of the 'context' (or narrower) variable.
    A 'c' has been added at the begging of the ID to indicate that
    we are refering to a context variable.
    """
    id_hash = hashlib.sha256((var['Variabele']['Id'] + var['LabelVanDeVariabele']).encode('utf-8')).hexdigest()
    narrower_variable_id = var_ns + "c" + id_hash
    return narrower_variable_id

In [10]:
def add_broader_variable_triples(var, broader_variable_id, narrower_variable_id, thesaurus_id):
    """
    Adding triples to the graph about the broader variables.
    """
    graph.add((URIRef(broader_variable_id), RDF.type, SKOS.Concept))
    graph.add((URIRef(broader_variable_id), SKOS.prefLabel, Literal(var['Variabele']['UniekeNaam'], lang='nl')))
    graph.add((URIRef(broader_variable_id), SKOS.definition, Literal(var['Variabele']['Definitie'], lang='nl')))
    graph.add((URIRef(broader_variable_id), SKOS.narrower, URIRef(narrower_variable_id)))
    graph.add((URIRef(broader_variable_id), SKOS.topConceptOf, URIRef(thesaurus_id)))
    graph.add((URIRef(broader_variable_id), SKOS.inScheme, URIRef(thesaurus_id)))
    add_narrower_variable_triples(narrower_variable_id, broader_variable_id)
    return graph

In [11]:
def add_narrower_variable_triples(narrower_variable_id, broader_variable_id):
    """
    Adding triples to the graph about the narrower variables.
    """
    graph.add((URIRef(narrower_variable_id), RDF.type, SKOS.Concept))
    graph.add((URIRef(narrower_variable_id), SKOS.prefLabel, Literal(var['LabelVanDeVariabele'], lang='nl')))
    graph.add((URIRef(narrower_variable_id), SKOS.altLabel, Literal(var['VerkorteSchrijfwijzeNaamVariabele'], lang='nl')))
    graph.add((URIRef(narrower_variable_id), SKOS.broader, URIRef(broader_variable_id)))
    return graph

In [12]:
def add_variable_triples(var, thesaurus_id):
    """
    Calling the adding variables functions.
    """
    broader_variable_id = make_broader_variable_id(var)
    narrower_variable_id = make_narrower_variable_id(var)
    
    graph.add((URIRef(thesaurus_id), SKOS.hasTopConcept, URIRef(broader_variable_id)))
    
    add_broader_variable_triples(var, broader_variable_id, narrower_variable_id, thesaurus_id)
    add_narrower_variable_triples(narrower_variable_id, broader_variable_id)
    
    return graph

In [13]:
def write_output_file(graph, ofile):
    """
    Write the content of the graph into an output file.
    """
    with open(ofile, "w") as f:
        f.write(graph.serialize(format="turtle"))

In [14]:
for file in os.listdir(directory_path):
    
    if file.endswith('.dsc'):
        
        thesaurus_id = make_cbs_variable_thesaurus_id(var_ns)
        
        define_cbs_variable_thesaurus(thesaurus_id)

        xml_dictionary = make_xml_dictionary(os.path.join(directory_path, file))
        
        variables_list = make_variables_list(xml_dictionary)
        
        for var in variables_list:
            add_variable_triples(var, thesaurus_id)
        
write_output_file(graph, ofile)